## Phantom Positions — Data Exploration

Exploratory analysis of the EMSCAD dataset (`fake_job_postings.csv`) prior to modeling.

**Covers:**
- Class distribution (real vs. fake postings)
- Missing values per feature column
- Top words in real vs. fake postings using CountVectorizer
- Words appearing exclusively in fake postings

In [ ]:
import sys
sys.path.append('..')
from data.preprocessing import TEXT_COLS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv("../data/fake_job_postings.csv")

summary = df['fraudulent'].value_counts().rename(index={0: "Real", 1: "Fake"})
percent = df['fraudulent'].value_counts(normalize=True).rename(index={0: "Real", 1: "Fake"}) * 100

result = pd.DataFrame({
    "Count": summary,
    "Percent (%)": percent.round(2)
})

print("Total:", len(df))
print(result)

In [ ]:
print("Original counts:")
print(df['fraudulent'].value_counts())

print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
counts = df['fraudulent'].value_counts()

plt.figure()
counts.plot(kind='bar')
plt.xticks([0, 1], ['Real', 'Fake'], rotation=0)
plt.title("Class Distribution")
plt.ylabel("Count")
plt.show()

print(counts)

In [ ]:
# Combine text
text_real = df[df['fraudulent'] == 0][TEXT_COLS].fillna('').agg(' '.join, axis=1)
text_fake = df[df['fraudulent'] == 1][TEXT_COLS].fillna('').agg(' '.join, axis=1)

vectorizer_real = CountVectorizer(stop_words='english', max_features=20)
vectorizer_fake = CountVectorizer(stop_words='english', max_features=20)

X_real = vectorizer_real.fit_transform(text_real)
real_counts = X_real.sum(axis=0).A1
real_words = vectorizer_real.get_feature_names_out()

X_fake = vectorizer_fake.fit_transform(text_fake)
fake_counts = X_fake.sum(axis=0).A1
fake_words = vectorizer_fake.get_feature_names_out()

In [ ]:
# Sort indices by descending count
sorted_idx = np.argsort(real_counts)[::-1]

# Apply sorting
real_words_sorted = real_words[sorted_idx]
real_counts_sorted = real_counts[sorted_idx]

plt.figure()
plt.bar(real_words_sorted, real_counts_sorted)
plt.title("Top Words in Real Job Postings")
plt.xticks(rotation=45)
plt.show()

# Sort indices by descending count
sorted_idx = np.argsort(fake_counts)[::-1]

# Apply sorting
fake_words_sorted = fake_words[sorted_idx]
fake_counts_sorted = fake_counts[sorted_idx]

plt.figure()
plt.bar(fake_words_sorted, fake_counts_sorted)
plt.title("Top Words in Fake Job Postings")
plt.xticks(rotation=45)
plt.show()

In [ ]:
import pandas as pd

real_df = pd.DataFrame({'word': real_words_sorted, 'count': real_counts_sorted.astype(int)})
fake_df = pd.DataFrame({'word': fake_words_sorted, 'count': fake_counts_sorted.astype(int)})

print("Top Words in Real Job Postings:")
print(real_df.to_string(index=False))

print("\nTop Words in Fake Job Postings:")
print(fake_df.to_string(index=False))

comparison_df = pd.DataFrame({
    'real_word': real_words_sorted,
    'real_count': real_counts_sorted.astype(int),
    'fake_word': fake_words_sorted,
    'fake_count': fake_counts_sorted.astype(int)
})

print(comparison_df.to_string(index=False))

In [ ]:
# Fit separate vectorizers on all text (not split by class) to get comparable counts
vectorizer_all = CountVectorizer(stop_words='english', max_features=20)
vectorizer_all.fit(pd.concat([text_real, text_fake]))

# Transform each class using the shared vocabulary
X_real_all = vectorizer_all.transform(text_real)
X_fake_all = vectorizer_all.transform(text_fake)

real_counts_all = X_real_all.sum(axis=0).A1
fake_counts_all = X_fake_all.sum(axis=0).A1
words_all = vectorizer_all.get_feature_names_out()

# Sort by fake count descending, take top 10
sorted_idx = np.argsort(fake_counts_all)[::-1][:10]

comparison_df = pd.DataFrame({
    'word': words_all[sorted_idx],
    'fake_count': fake_counts_all[sorted_idx].astype(int),
    'real_count': real_counts_all[sorted_idx].astype(int),
    'pct_in_real': (real_counts_all[sorted_idx] / (real_counts_all[sorted_idx] + fake_counts_all[sorted_idx]) * 100).round(1),
    'pct_in_fake': (fake_counts_all[sorted_idx] / (real_counts_all[sorted_idx] + fake_counts_all[sorted_idx]) * 100).round(1)
})

print(comparison_df.to_string(index=False))

In [ ]:
vectorizer_unique = CountVectorizer(stop_words='english', max_features=5000)
vectorizer_unique.fit(pd.concat([text_real, text_fake]))

X_real_u = vectorizer_unique.transform(text_real)
X_fake_u = vectorizer_unique.transform(text_fake)

real_counts_u = X_real_u.sum(axis=0).A1
fake_counts_u = X_fake_u.sum(axis=0).A1
words_u = vectorizer_unique.get_feature_names_out()

# Words with zero real count but nonzero fake count
mask = (real_counts_u == 0) & (fake_counts_u > 0)
fake_only_df = pd.DataFrame({
    'word': words_u[mask],
    'fake_count': fake_counts_u[mask].astype(int)
}).sort_values('fake_count', ascending=False)

print(f"Words appearing only in fake postings: {len(fake_only_df)}")
print(fake_only_df.to_string(index=False))

In [ ]:
aker_mask = df[TEXT_COLS].fillna('').apply(lambda col: col.str.contains('aker', case=False)).any(axis=1)
subsea_mask = df[TEXT_COLS].fillna('').apply(lambda col: col.str.contains('subsea', case=False)).any(axis=1)

both_mask = aker_mask & subsea_mask & (df['fraudulent'] == 1)
aker_only = aker_mask & ~subsea_mask & (df['fraudulent'] == 1)
subsea_only = subsea_mask & ~aker_mask & (df['fraudulent'] == 1)

print(f"Fake postings with both 'aker' and 'subsea': {both_mask.sum()}")
print(f"Fake postings with 'aker' only: {aker_only.sum()}")
print(f"Fake postings with 'subsea' only: {subsea_only.sum()}")